# Unmasked — Re-identification Risk (Role A charts)

These charts read **only** the aggregate result files in `results/` (CSV/JSON).
No row-level student data is loaded here, in line with the project's privacy rule.
Each chart is saved to `results/charts/*.png` at 2x scale for the slides.

Accessibility: every bar/cell is labelled, axes are formatted as percentages,
and the colorblind-safe palette `px.colors.qualitative.Safe` is used throughout.

In [1]:
import json
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Resolve the repo root by walking up until we find config.yaml, so the notebook
# works whether it is run from the repo root or the notebooks/ folder (nbconvert,
# Jupyter, VS Code all set different working directories).
ROOT = Path.cwd()
for _ in range(5):
    if (ROOT / "config.yaml").exists():
        break
    ROOT = ROOT.parent
RESULTS = ROOT / "results"
CHARTS = RESULTS / "charts"
CHARTS.mkdir(parents=True, exist_ok=True)

SAFE = px.colors.qualitative.Safe          # colorblind-safe qualitative palette
ACCENT = SAFE[1]                            # single accent colour for one-series bars

# Plain-language names for quasi-identifier columns, used across the charts.
PRETTY = {
    "MAJfirstcode": "First major",
    "MAJsecondcode": "Second major",
    "begincol": "Began at college",
    "edaspire": "Education goal",
    "FTFY": "First-time first-year",
    "affiliation": "Affiliation",
    "program": "Program",
    "Year": "Year",
    "Class": "Class",
}


def save(fig, name):
    path = CHARTS / name
    fig.write_image(str(path), scale=2)     # needs kaleido
    print("saved", path)
    return fig


## 1. Each added field makes more students unique

**Headline:** using just 8 ordinary fields, **67.8%** of students become uniquely
identifiable; add the five prior-attendance flags (14 fields) and it reaches **73.9%**.

In [2]:
st = pd.read_csv(RESULTS / "staircase.csv")

# Short, readable x labels in file order; full field list stays in hover.
SHORT = {
    2: "Year + Class",
    3: "+ affiliation",
    4: "+ first major",
    5: "+ program",
    6: "+ second major",
    8: "8 core fields",
    14: "14 fields",
}
st["label"] = st["n_fields"].map(SHORT).fillna(st["n_fields"].astype(str) + " fields")

fig1 = px.bar(
    st, x="label", y="pct_unique", text_auto=".1%",
    color_discrete_sequence=[ACCENT],
    hover_data={"fields": True, "pct_unique": ":.1%", "pct_below_k": ":.1%", "label": False},
    title="Each ordinary field you add makes more students unique",
)
fig1.update_traces(textposition="outside", cliponaxis=False)
fig1.update_layout(
    xaxis_title="", yaxis_title="% uniquely identifiable",
    yaxis_tickformat=".0%", yaxis_range=[0, 0.8],
    plot_bgcolor="white", bargap=0.25,
)
save(fig1, "staircase.png")
fig1


saved C:\Users\mcbri\Documents\DS Hackathon\unmasked\results\charts\staircase.png


## 2. The riskiest pairs of fields

**Headline:** on their own, two fields rarely make anyone fully unique, but some
pairs push a large share of students into groups smaller than k=5. The heatmap
colours each pair by **% of students in groups below k=5** (the real exposure).

In [3]:
combos = pd.read_csv(RESULTS / "combos.csv")
pairs = combos[combos["size"] == 2].copy()
# regex=False: the " + " separator contains a literal '+', which pandas would
# otherwise treat as a regex quantifier and fail to split on.
split = pairs["combo"].str.split(" + ", expand=True, regex=False)
pairs["a"], pairs["b"] = split[0], split[1]

# Build a symmetric matrix of pct_below_k, with plain-language field labels.
raw_fields = sorted(set(pairs["a"]) | set(pairs["b"]))
mat = pd.DataFrame(index=raw_fields, columns=raw_fields, dtype=float)
for _, r in pairs.iterrows():
    mat.loc[r["a"], r["b"]] = r["pct_below_k"]
    mat.loc[r["b"], r["a"]] = r["pct_below_k"]

labels = [PRETTY.get(f, f) for f in raw_fields]
top = pairs.sort_values("pct_below_k", ascending=False).iloc[0]
top_a, top_b = PRETTY.get(top["a"], top["a"]), PRETTY.get(top["b"], top["b"])
text = mat.map(lambda v: "" if pd.isna(v) else f"{v:.0%}")  # DataFrame.map (pandas >= 2.1)

fig2 = go.Figure(go.Heatmap(
    z=mat.values, x=labels, y=labels,
    text=text.values, texttemplate="%{text}",
    colorscale="Cividis",                      # sequential colorblind-safe
    colorbar=dict(title="% below k=5", tickformat=".0%"),
    hovertemplate="%{y} + %{x}: %{z:.1%}<extra></extra>",
))
fig2.update_layout(
    title=f"Riskiest pair: {top_a} + {top_b} puts {top['pct_below_k']:.0%} of students in groups below k=5",
    xaxis_title="", yaxis_title="", plot_bgcolor="white",
)
save(fig2, "riskiest_pairs.png")
fig2


saved C:\Users\mcbri\Documents\DS Hackathon\unmasked\results\charts\riskiest_pairs.png


## 3. What the attacker knows decides who they can single out

**Headline:** a classmate (year, class, major) can uniquely pick out ~1% of
students, but someone who knows all core fields can single out **67.8%**.

In [4]:
per = pd.read_csv(RESULTS / "personas.csv")
NICE = {
    "classmate": "Classmate",
    "club_roster": "Club roster",
    "linkedin": "LinkedIn",
    "survey_insider": "Survey insider",
    "everything": "Knows everything",
}
per["name"] = per["persona"].map(NICE).fillna(per["persona"])
per = per.sort_values("pct_unique")      # ascending for horizontal bars

fig3 = px.bar(
    per, x="pct_unique", y="name", orientation="h", text_auto=".1%",
    color_discrete_sequence=[ACCENT],
    hover_data={"fields": True, "pct_unique": ":.1%", "name": False},
    # Short, wrapped title so it is never clipped.
    title="What an attacker knows decides<br>how many students they can single out",
)
fig3.update_traces(textposition="outside", cliponaxis=False)
fig3.update_layout(
    xaxis_title="% of students uniquely identifiable", yaxis_title="",
    xaxis_tickformat=".0%", xaxis_range=[0, 0.75], plot_bgcolor="white",
    margin=dict(t=70),
)
save(fig3, "personas.png")
fig3


saved C:\Users\mcbri\Documents\DS Hackathon\unmasked\results\charts\personas.png


## 4. The smallest groups are the most exposed

**Headline:** on the core fields, **98.8%** of Gray mouse lemur students (the
smallest group, n=864) fall in groups below k=5, versus **84.7%** of the largest
group (Ring-tailed lemur, n=10,460). The smallest groups carry the highest risk.

In [5]:
rbg = pd.read_csv(RESULTS / "risk_by_group.csv")
rbg = rbg.sort_values("pct_below_k", ascending=True)
rbg["glabel"] = rbg["group"] + " (n=" + rbg["n"].astype(int).astype(str) + ")"

fig4 = px.bar(
    rbg, x="pct_below_k", y="glabel", orientation="h", text_auto=".1%",
    color_discrete_sequence=[SAFE[3]],
    hover_data={"n": True, "pct_unique": ":.1%", "pct_below_k": ":.1%", "glabel": False},
    title="The smallest groups are the most at risk of re-identification",
)
fig4.update_traces(textposition="outside", cliponaxis=False)
fig4.update_layout(
    xaxis_title="% of students in groups smaller than k=5", yaxis_title="",
    xaxis_tickformat=".0%", xaxis_range=[0, 1.05], plot_bgcolor="white",
)
save(fig4, "risk_by_group.png")
fig4


saved C:\Users\mcbri\Documents\DS Hackathon\unmasked\results\charts\risk_by_group.png


## 5. Attack before / after (needs Role B's `mitigate.py`)

Once Role B's `mitigate.py` is merged and `scripts/run_experiments.py` writes
`results/attack.json`, this section charts how the simulated linkage attack drops
from ~68.5% re-identified to 0% after mitigation. Until then it prints a note.

In [6]:
attack_path = RESULTS / "attack.json"
if attack_path.exists():
    data = json.load(open(attack_path))
    mode = data.get("protect_on", next(iter(data.values())))
    rows = [{"persona": p, "before": v["before"]["correct_rate"],
             "after": v["after"]["correct_rate"]} for p, v in mode.items()]
    ba = pd.DataFrame(rows).melt(id_vars="persona", var_name="stage", value_name="rate")
    figb = px.bar(
        ba, x="persona", y="rate", color="stage", barmode="group", text_auto=".0%",
        color_discrete_sequence=SAFE,
        title="Simulated linkage attack: correctly re-identified, before vs after mitigation",
    )
    figb.update_layout(yaxis_tickformat=".0%", yaxis_title="% correctly re-identified",
                       xaxis_title="", plot_bgcolor="white")
    save(figb, "attack_before_after.png")
    figb
else:
    print("results/attack.json not found yet. Run scripts/run_experiments.py once "
          "Role B's mitigate.py and utility.py are available to generate it.")


results/attack.json not found yet. Run scripts/run_experiments.py once Role B's mitigate.py and utility.py are available to generate it.
